# Task 3 di Kaggle: Retrieval (RQ3) + Stance (RQ4)
Kode ditarik dari **GitHub**, checkpoint NER dan data tambahan dari **Google Drive** (`gdown`), Komdigi dari **Kaggle Dataset**.
Kaggle tidak punya `drive.mount`, jadi folder Drive harus dibagikan "Anyone with the link" (Viewer).

Sebelum mulai:
1. Settings → Accelerator **GPU T4 x2 / P100**, Internet **On**.
2. Add Input → cari dataset `ireddragonicy/indonesian-hoax-news-dataset` (harus versi 396, sama dengan anotasi tim).
3. (Repo privat saja) Add-ons → Secrets → `GITHUB_TOKEN`.
4. Kode task 3 harus sudah ada di `GITHUB_BRANCH` (default `main`).

In [ ]:
# ===== KONFIGURASI: ubah di sini =====
GITHUB_REPO   = "danenftyessir/indonesian-hoax-checker"
GITHUB_BRANCH = "main"
# Folder Drive tim (dokumen). Hanya subfolder `data/` dan `artifacts/` di dalamnya yang disalin ke repo.
GDRIVE_DATA_FOLDER_ID = "1rregnX8Qq9pU7rlTB7QUiXubWVJ5-zS0"
# Folder hasil_training: berisi ner_<waktu>/ner_results.zip (checkpoint NER dari Jonathan)
GDRIVE_NER_FOLDER_ID  = "1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn"
NER_CHECKPOINT = "artifacts/checkpoints/ner/ner_indobert_seed42"
KAGGLE_INPUT_GLOB = "/kaggle/input/*/komdigi_hoaks.csv"
RUN_STAGE1_TRAINING = True   # fine-tune IndoBERT di IndoNLI (~20-40 menit di T4)

In [ ]:
import os, subprocess, glob, shutil, pathlib
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token+'@' if token else ''}github.com/{GITHUB_REPO}.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/indonesian-hoax-checker"
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","-b",GITHUB_BRANCH,url,REPO],check=True)
os.chdir(REPO)
!git log --oneline -3

In [ ]:
# extras [training] = datasets, accelerate, seqeval, pytorch-crf (dibutuhkan ner.predict untuk entitas)
!pip -q install -e ".[training]" gdown 2>&1 | tail -3
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU belum aktif: Settings -> Accelerator -> GPU"

## Data
Komdigi dari Kaggle Input, IndoNLI dari GitHub raw, tambahan (BRIN, checkpoint, dll.) dari Drive.

In [ ]:
import yaml
os.makedirs("data/raw/refuted_claims", exist_ok=True)
hits = glob.glob(KAGGLE_INPUT_GLOB)
assert hits, "Dataset Komdigi belum ditambahkan sebagai Input"
shutil.copy(hits[0], "data/raw/refuted_claims/komdigi_hoaks.csv")

src = yaml.safe_load(open("configs/data_sources.yaml"))["indonli"]
os.makedirs(src["local_raw"], exist_ok=True)
for f in src["files"]:
    !wget -q -nc -P {src["local_raw"]} {src["base_url"]}{f}
!ls -la data/raw/refuted_claims data/raw/stance/indonli

In [ ]:
import gdown, zipfile

if GDRIVE_DATA_FOLDER_ID:
    try:
        gdown.download_folder(id=GDRIVE_DATA_FOLDER_ID, output="/kaggle/working/gdrive_in", quiet=True)
        for rel in ["data", "artifacts"]:
            p = pathlib.Path("/kaggle/working/gdrive_in")/rel
            if p.exists(): shutil.copytree(p, rel, dirs_exist_ok=True); print("disalin:", rel)
    except Exception as e:
        print("Folder data Drive gagal diunduh (bagikan 'Anyone with the link'?):", e)

# Checkpoint NER: pilih ner_results.zip dari subfolder ner_* terbaru, unduh satu file saja
if GDRIVE_NER_FOLDER_ID and not os.path.exists(NER_CHECKPOINT):
    try:
        files = gdown.download_folder(id=GDRIVE_NER_FOLDER_ID, skip_download=True, quiet=True, remaining_ok=True)
        zips = sorted([f for f in files if f.path.replace(chr(92), "/").endswith("/ner_results.zip")
                       and "/rq1_" not in f.path.replace(chr(92), "/")], key=lambda f: f.path)
        assert zips, "ner_results.zip tidak ditemukan di folder hasil_training"
        z = zips[-1]; print("memakai", z.path)
        gdown.download(id=z.id, output="/kaggle/working/ner_results.zip", quiet=False)
        zipfile.ZipFile("/kaggle/working/ner_results.zip").extractall(".")
    except Exception as e:
        print("Checkpoint NER gagal diunduh; ablasi filter entitas dilewati:", e)
print("checkpoint NER ada:", os.path.exists(NER_CHECKPOINT))

## Fase 1: klaim, split, konversi stance

In [ ]:
!python -m indonesian_hoax_checker.retrieval.corpus data/raw/refuted_claims/komdigi_hoaks.csv data/interim/refuted_claims/claims_raw.jsonl
!python -m indonesian_hoax_checker.evaluation.splits data/interim/refuted_claims/claims_raw.jsonl --kaggle-version 396
for s in ["train","val","test_lay","test_expert"]:
    !python -m indonesian_hoax_checker.stance.dataset indonli data/raw/stance/indonli/{s}.jsonl data/processed/stance/indonli_{s}.jsonl --split-name {s}

# Anotasi tim (kueri retrieval, pasangan stance) mengacu ke claim_id Komdigi v396. Bila versi Kaggle Input
# berbeda, gold id bisa hilang dan metrik menyesatkan: hentikan di sini.
import json
ids = {json.loads(l)["claim_id"] for l in open("data/processed/refuted_claims/claims.jsonl", encoding="utf-8")}
gold = {g for l in open("data/annotations/claim_verification/retrieval_queries.jsonl", encoding="utf-8") if l.strip()
        for g in json.loads(l)["gold_claim_ids"]}
missing = gold - ids
assert not missing, f"{len(missing)} gold_claim_id tidak ada di basis klaim: versi dataset Kaggle Input berbeda dari v396"
print("gold id cocok:", len(gold))

## RQ3: retrieval (BM25 vs E5, tanpa/dengan filter entitas)

In [ ]:
Q = "data/annotations/claim_verification/retrieval_queries.jsonl"
have_ner = os.path.exists(NER_CHECKPOINT)
if have_ner:
    # entitas PREDIKSI NER (keputusan 0005) untuk basis klaim dan kueri
    !python -m indonesian_hoax_checker.retrieval.annotate_entities --checkpoint {NER_CHECKPOINT} --queries {Q}
runs = [("rq3_bm25_nofilter","configs/models/retrieval_bm25.yaml",""),
        ("rq3_e5_nofilter","configs/models/retrieval_embeddings.yaml","")]
if have_ner:
    runs += [("rq3_bm25_entfilter","configs/models/retrieval_bm25.yaml","--entity-filter"),
             ("rq3_e5_entfilter","configs/models/retrieval_embeddings.yaml","--entity-filter")]
for name,cfg,flag in runs:
    print("=== ",name)
    r = subprocess.run(f"python -m indonesian_hoax_checker.retrieval.evaluate {cfg} --queries {Q} --run-name {name} {flag}",shell=True)
    if r.returncode: print("GAGAL:", name)

## RQ4: stance

In [ ]:
if RUN_STAGE1_TRAINING:
    !python -m indonesian_hoax_checker.stance.train configs/models/stance_nli.yaml --train data/processed/stance/indonli_train.jsonl --dev data/processed/stance/indonli_val.jsonl --output artifacts/checkpoints/stance_indonli --seed 42

In [ ]:
models = [("rq4_mdeberta_zeroshot","MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")]
if os.path.exists("artifacts/checkpoints/stance_indonli"): models.append(("rq4_indobert_indonli","artifacts/checkpoints/stance_indonli"))
if os.path.exists("artifacts/checkpoints/stance_final"):   models.append(("rq4_indobert_final","artifacts/checkpoints/stance_final"))
for split in ["indonli_test_expert","indonli_val"]:
    for name,m in models:
        !python -m indonesian_hoax_checker.stance.predict --model {m} --pairs data/processed/stance/{split}.jsonl --run-name {name}_{split}

In [ ]:
import json
for f in sorted(glob.glob("outputs/metrics/*.json")):
    print(f); print(json.dumps(json.load(open(f)),indent=1,ensure_ascii=False)[:800])

## Simpan hasil
Notebook tidak bisa menulis ke Drive tanpa kredensial. Zip di bawah, lalu tab *Output* → Download, unggah manual ke Drive.
`claims_entities.jsonl` dan `retrieval_queries.jsonl` (entitas terisi) ada di dalam zip: salin kembali ke repo lokal agar ablasi bisa direproduksi.

In [ ]:
!cd /kaggle/working && zip -rq task3_results.zip indonesian-hoax-checker/outputs indonesian-hoax-checker/artifacts/checkpoints/stance_indonli indonesian-hoax-checker/data/splits indonesian-hoax-checker/data/processed/refuted_claims/claims_entities.jsonl indonesian-hoax-checker/data/annotations/claim_verification/retrieval_queries.jsonl
!ls -lh /kaggle/working/task3_results.zip